# Notebook 01 — เริ่มต้น Google Earth Engine ด้วย Python แบบทีละขั้น
## กรณีศึกษา: กรุงเทพมหานคร (Bangkok)

Notebook นี้ออกแบบสำหรับนิสิตที่มีพื้นฐาน Python เพียงเล็กน้อย โดยเน้น **รันทีละ cell → ดูผลบนแผนที่ → ทำความเข้าใจแนวคิด → ทดลองแก้ค่าเล็กน้อย**

### คำถามหลัก
> เราจะใช้ Google Earth Engine (GEE) เพื่อเลือกขอบเขตกรุงเทพมหานคร เลือกภาพ Sentinel-2 กรองเวลาและเมฆ แสดงภาพหลายรูปแบบ คำนวณ NDVI อ่านค่าจากจุด สุ่มตัวอย่าง และส่งออก GeoTIFF ได้อย่างไร?

### เมื่อจบ Notebook นี้ นิสิตควรสามารถ
- เข้าใจความหมายของ `Map`, `Layer`, `Geometry`, `FeatureCollection`, `Image`, `ImageCollection`
- ใช้ FAO GAUL เพื่อเลือกขอบเขตกรุงเทพมหานคร
- ใช้ `filterBounds()`, `filterDate()` และ cloud metadata เพื่อคัดเลือกภาพ
- แสดง Sentinel-2 แบบ True Color และ False Color
- สร้าง median composite และ clip ตามขอบเขตกรุงเทพฯ
- คำนวณ NDVI อย่างง่าย
- อ่านค่า NDVI ณ ตำแหน่งหนึ่งและหลายตำแหน่ง
- สุ่มจุดและดึงค่าจาก raster เป็นตาราง
- ส่งออก NDVI เป็น GeoTIFF และ sample table เป็น CSV

---

## แนวคิดของทั้ง Notebook

```text
Map
 ↓
Point / Polygon
 ↓
GAUL boundary
 ↓
Sentinel-2 ImageCollection
 ↓
filterBounds
 ↓
filterDate
 ↓
filter cloud
 ↓
Image / Composite
 ↓
RGB / False Color / NDVI
 ↓
Point query / Random sampling
 ↓
GeoTIFF / CSV
```

> **หลักการสำคัญ:** ใน Notebook นี้ “หนึ่ง Code Cell = หนึ่งแนวคิด” เพื่อให้เห็นว่าคำสั่งแต่ละชุดเปลี่ยนข้อมูลหรือแผนที่อย่างไร

# Part A — เตรียมระบบและทำความรู้จักแผนที่

## Step 1 — Import library และเชื่อมต่อ Earth Engine

เราจะใช้เพียง library หลักที่จำเป็นก่อน:

- `ee` = Google Earth Engine Python API
- `geemap` = เครื่องมือแสดงผล Earth Engine บน interactive map
- `pandas` = ใช้กับตารางข้อมูลขนาดเล็กในช่วงท้าย

`PROJECT` คือ Google Cloud Project ID ที่ลงทะเบียนใช้งาน Earth Engine แล้ว

In [1]:
%pip install -q earthengine-api geemap pandas

import ee
import geemap
import pandas as pd

# Project ID ตัวอย่างของรายวิชา
# หากนิสิตใช้ Project ของตนเอง ให้เปลี่ยนค่าในบรรทัดนี้
PROJECT = "project-cf559859-d42a-44d4-925"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized successfully.")
print("Using project:", PROJECT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 37.9 MB/s eta 0:00:00
Earth Engine initialized successfully.
Using project: project-cf559859-d42a-44d4-925


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## Step 2 — สร้างฟังก์ชันแผนที่มาตรฐานของรายวิชา

ใน Colab environment นี้ default OpenStreetMap tile อาจถูก block จึงสร้างฟังก์ชัน `create_course_map()` เพื่อ:

1. สร้าง `geemap.Map()`
2. ลบ default basemap
3. ใช้ `Esri.WorldImagery` เป็นพื้นหลังเริ่มต้น
4. เตรียม `Esri.WorldStreetMap` และ `Esri.WorldTopoMap` ไว้ให้เปิดจาก Layer Control

> **Basemap** คือแผนที่พื้นหลังเพื่อช่วยอ้างอิงตำแหน่ง  
> **Thematic layer** คือข้อมูลที่เรากำลังวิเคราะห์ เช่น ขอบเขตกรุงเทพฯ, Sentinel-2 หรือ NDVI

In [2]:
def create_course_map():
    m = geemap.Map()
    m.clear_layers()

    m.add_basemap("Esri.WorldImagery", show=True)
    m.add_basemap("Esri.WorldStreetMap", show=False)
    m.add_basemap("Esri.WorldTopoMap", show=False)

    return m

## Step 3 — แสดงแผนที่เปล่า

ลอง zoom, pan และเปิด/ปิด basemap

### Checkpoint
- `WorldImagery` เหมาะกับการดูพื้นผิวจริงอย่างไร?
- `WorldStreetMap` ช่วยเรื่องโครงสร้างเมืองอย่างไร?
- `WorldTopoMap` เหมาะกับพื้นที่ภูเขาอย่างไร?

In [3]:
Map = create_course_map()
Map.add_layer_control()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# Part B — จากพิกัดไปสู่ขอบเขตกรุงเทพมหานคร

## Step 4 — สร้าง Point ของกรุงเทพมหานคร

Earth Engine ใช้ลำดับพิกัด:

```text
[longitude, latitude]
```

ตัวอย่างจุดใกล้ใจกลางกรุงเทพฯ:

```text
Longitude = 100.50
Latitude  = 13.75
```

จุด (`Point`) เป็น geometry ที่ง่ายที่สุดก่อนเรียน polygon

In [4]:
bangkok_point = ee.Geometry.Point([100.50, 13.75])

Map = create_course_map()
Map.centerObject(bangkok_point, 11)

Map.addLayer(
    bangkok_point,
    {"color": "yellow"},
    "Bangkok point"
)

Map.add_layer_control()
Map

Map(center=[13.75, 100.50000000000001], controls=(WidgetControl(options=['position', 'transparent_bg'], positi…

### Exercise 1 — ทดลองเปลี่ยนพิกัด

แก้ longitude/latitude เพื่อทดลองไปยังสถานที่อื่น เช่น:

- Chiang Mai ประมาณ `[98.99, 18.79]`
- Phitsanulok ประมาณ `[100.26, 16.82]`

**คำถาม:** เมื่อเปลี่ยน coordinate แล้ว จุดบนแผนที่เคลื่อนไปตามที่คาดหรือไม่?

## Step 5 — เรียกข้อมูลขอบเขตการปกครอง FAO GAUL

ใน Desktop GIS เรามักใช้ข้อมูล vector ในรูป Shapefile (`.shp`)  
ใน Earth Engine ข้อมูล vector มักอยู่ในรูป `FeatureCollection`

ตัวอย่างนี้ใช้:

`FAO/GAUL/2025/level1`

ซึ่งเป็นขอบเขตการปกครองระดับที่ 1 เช่น จังหวัดหรือหน่วยเทียบเท่า

field สำคัญ:
- `GAUL0_NAME` = ชื่อประเทศ
- `GAUL1_NAME` = ชื่อหน่วยปกครองระดับที่ 1

> สำหรับการเรียน ให้คิดว่า `FeatureCollection` นี้ทำหน้าที่คล้าย “ชั้นข้อมูล shapefile” ที่ถูกเก็บไว้บน Earth Engine

In [5]:
admin1 = ee.FeatureCollection("FAO/GAUL/2025/level1")

print(admin1)

ee.FeatureCollection({
  "functionInvocationValue": {
    "functionName": "Collection.loadTable",
    "arguments": {
      "tableId": {
        "constantValue": "FAO/GAUL/2025/level1"
      }
    }
  }
})


## Step 6 — Filter ประเทศไทยก่อน

เราจะค่อย ๆ ลดข้อมูลจาก “ทั่วโลก” เหลือ “ประเทศไทย”

นี่คือแนวคิดเดียวกับ Attribute Query ใน GIS

In [6]:
thailand = admin1.filter(
    ee.Filter.eq("GAUL0_NAME", "Thailand")
)

print("จำนวนหน่วยปกครองระดับ 1 ของประเทศไทย:",
      thailand.size().getInfo())

จำนวนหน่วยปกครองระดับ 1 ของประเทศไทย: 77


## Step 7 — ตรวจสอบ Attribute ก่อนเลือกกรุงเทพฯ

ก่อนใช้ `filter()` ควรตรวจสอบชื่อ field และค่าที่อยู่ในข้อมูลก่อน

เราจะดูตัวอย่าง attribute ของ feature แรก และดูชื่อจังหวัดบางส่วน

In [7]:
first_province = ee.Feature(thailand.first())

print("ตัวอย่าง attribute:")
print(first_province.toDictionary().getInfo())

province_names = thailand.aggregate_array("GAUL1_NAME").sort().getInfo()

print("\nตัวอย่างชื่อจังหวัด 15 รายการแรก:")
print(province_names[:15])

ตัวอย่าง attribute:
{'CONTINENT': 'Asia', 'DISP_EN': 'Lampang, Thailand', 'FID': 2876, 'GAUL0_CODE': 275, 'GAUL0_NAME': 'Thailand', 'GAUL1_CODE': 3025, 'GAUL1_NAME': 'Lampang', 'ISO3_CODE': 'THA', 'MAP_CODE': 'THA'}

ตัวอย่างชื่อจังหวัด 15 รายการแรก:
['Amnat Charoen', 'Ang Thong', 'Bangkok', 'Bueng Kan', 'Buri Ram', 'Chachoengsao', 'Chai Nat', 'Chaiyaphum', 'Chanthaburi', 'Chiang Mai', 'Chiang Rai', 'Chon Buri', 'Chumphon', 'Kalasin', 'Kamphaeng Phet']


## Step 8 — แสดงทุกจังหวัดของประเทศไทยบนแผนที่

ตอนนี้เรามี `FeatureCollection` ของประเทศไทยแล้ว

สังเกตว่า vector layer สามารถถูก style และแสดงบน basemap ได้เหมือน shapefile ใน GIS

In [8]:
thailand_outline = thailand.style(
    color="FFFF00",
    fillColor="00000000",
    width=1
)

Map = create_course_map()
Map.centerObject(thailand, 5)

Map.addLayer(
    thailand_outline,
    {},
    "Thailand provinces"
)

Map.add_layer_control()
Map

Map(center=[15.066681977563354, 101.01112483785317], controls=(WidgetControl(options=['position', 'transparent…

## Step 9 — Filter กรุงเทพมหานคร

จากทุกจังหวัดของประเทศไทย เราเลือกเฉพาะ:

```text
GAUL1_NAME = Bangkok
```

In [9]:
bangkok = thailand.filter(
    ee.Filter.eq("GAUL1_NAME", "Bangkok")
)

print("Bangkok features:", bangkok.size().getInfo())

Bangkok features: 1


## Step 10 — แสดงขอบเขตกรุงเทพมหานคร

ตอนนี้ `bangkok` เป็น `FeatureCollection` ที่มี polygon ของกรุงเทพมหานคร

เราจะกำหนด fill ให้โปร่งใสเพื่อให้มองเห็น basemap ด้านล่าง

In [10]:
bangkok_outline = bangkok.style(
    color="FFFF00",
    fillColor="00000000",
    width=3
)

Map = create_course_map()
Map.centerObject(bangkok, 9)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok administrative boundary"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677679, 100.62341403243893], controls=(WidgetControl(options=['position', 'transparent…

### Checkpoint — อ่านขอบเขตเชิงพื้นที่

1. ขอบเขต administrative boundary ของกรุงเทพฯ ครอบคลุมพื้นที่แบบใดบ้าง?
2. พื้นที่ทุกส่วนภายใน boundary มีลักษณะเป็น built-up หนาแน่นเหมือนกันหรือไม่?
3. เหตุใด administrative boundary จึงไม่เท่ากับ physical urban extent เสมอ?

## Step 11 — เปลี่ยน FeatureCollection เป็น Geometry

สำหรับการ filter และ clip raster เรามักต้องใช้ geometry ของพื้นที่ศึกษา

คำว่า **ROI (Region of Interest)** หมายถึงพื้นที่ที่เราสนใจวิเคราะห์

In [11]:
bangkok_roi = bangkok.geometry()

print(bangkok_roi)

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Collection.geometry",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.filter",
          "arguments": {
            "collection": {
              "functionInvocationValue": {
                "functionName": "Collection.filter",
                "arguments": {
                  "collection": {
                    "functionInvocationValue": {
                      "functionName": "Collection.loadTable",
                      "arguments": {
                        "tableId": {
                          "constantValue": "FAO/GAUL/2025/level1"
                        }
                      }
                    }
                  },
                  "filter": {
                    "functionInvocationValue": {
                      "functionName": "Filter.equals",
                      "arguments": {
                        "leftField": {
                   

# Part C — เริ่มใช้ Sentinel-2 แบบทีละขั้น

## Step 12 — เรียก Sentinel-2 ImageCollection

Dataset:

`COPERNICUS/S2_SR_HARMONIZED`

Band ที่ใช้ใน Notebook นี้:

| Band | ความหมาย | Resolution |
|---|---|---:|
| B2 | Blue | 10 m |
| B3 | Green | 10 m |
| B4 | Red | 10 m |
| B8 | Near Infrared (NIR) | 10 m |

`ImageCollection` หมายถึงชุดภาพหลาย scene / หลายวัน ไม่ใช่ภาพเดียว

In [12]:
s2_all = ee.ImageCollection(
    "COPERNICUS/S2_SR_HARMONIZED"
)

print(s2_all)

ee.ImageCollection({
  "functionInvocationValue": {
    "functionName": "ImageCollection.load",
    "arguments": {
      "id": {
        "constantValue": "COPERNICUS/S2_SR_HARMONIZED"
      }
    }
  }
})


## Step 13 — Filter ตามพื้นที่: `filterBounds()`

จาก Sentinel-2 จำนวนมากทั่วโลก เราเลือกเฉพาะ scene ที่ตัดกับกรุงเทพมหานคร

In [13]:
s2_bkk_space = s2_all.filterBounds(bangkok_roi)

print("จำนวนภาพที่ตัดกับกรุงเทพฯ:",
      s2_bkk_space.size().getInfo())

จำนวนภาพที่ตัดกับกรุงเทพฯ: 1838


## Step 14 — Filter ตามเวลา: `filterDate()`

เรากำหนดช่วงเวลาเพื่อให้คำถามการวิเคราะห์ชัดเจน

ตัวอย่างนี้ใช้ช่วง:

**1 พฤศจิกายน 2024 – 1 มีนาคม 2025**

หมายเหตุ: `filterDate(start, end)` ใช้ `end` แบบ exclusive

In [14]:
START = "2024-11-01"
END   = "2025-03-01"

s2_bkk_time = (
    s2_bkk_space
    .filterDate(START, END)
)

print("จำนวนภาพหลัง filterDate:",
      s2_bkk_time.size().getInfo())

จำนวนภาพหลัง filterDate: 75


## Step 15 — ดูค่า Cloud Percentage ก่อนกรอง

Sentinel-2 มี metadata ชื่อ:

`CLOUDY_PIXEL_PERCENTAGE`

ค่านี้เป็นสัดส่วนเมฆของ scene/granule ไม่ใช่ cloud mask ราย pixel

เราจะดูค่าบางส่วนก่อนตัดสินใจ filter

In [15]:
cloud_values = (
    s2_bkk_time
    .aggregate_array("CLOUDY_PIXEL_PERCENTAGE")
    .getInfo()
)

print("Cloud percentage ของ 10 scene แรก:")
print([round(x, 1) for x in cloud_values[:10]])

Cloud percentage ของ 10 scene แรก:
[5.5, 6.8, 21.3, 100.0, 95.7, 91.4, 0.2, 0.1, 0.1, 5.7]


## Step 16 — Filter scene ที่เมฆน้อยกว่า 20%

ตอนนี้เรากำหนด:

```text
CLOUDY_PIXEL_PERCENTAGE < 20
```

และเปรียบเทียบจำนวนภาพก่อนและหลัง filter

In [16]:
CLOUD_THRESHOLD = 20

s2_bkk_cloud = s2_bkk_time.filter(
    ee.Filter.lt(
        "CLOUDY_PIXEL_PERCENTAGE",
        CLOUD_THRESHOLD
    )
)

print("ก่อนกรองเมฆ:", s2_bkk_time.size().getInfo())
print("หลังกรองเมฆ:", s2_bkk_cloud.size().getInfo())

ก่อนกรองเมฆ: 75
หลังกรองเมฆ: 34


### Exercise 2 — ทดลอง Cloud Threshold

ลองเปลี่ยน `CLOUD_THRESHOLD` เป็น:

- 10
- 20
- 40

แล้วบันทึกจำนวนภาพที่เหลือ

| Cloud threshold | จำนวนภาพ |
|---:|---:|
| 10% | ... |
| 20% | ... |
| 40% | ... |

**คำถาม:** threshold ที่เข้มขึ้นทำให้จำนวนภาพเพิ่มหรือลด เพราะเหตุใด?

# Part D — จาก ImageCollection ไปสู่ Image

## Step 17 — เลือกภาพเดียวที่มีเมฆน้อยที่สุด

ก่อนสร้าง composite เราจะดู “ภาพเดียว” ก่อน เพื่อเข้าใจว่าแต่ละ scene ใน `ImageCollection` มีลักษณะอย่างไร

เรา sort ด้วย `CLOUDY_PIXEL_PERCENTAGE` แล้วเลือกภาพแรก

In [17]:
best_scene = ee.Image(
    s2_bkk_cloud
    .sort("CLOUDY_PIXEL_PERCENTAGE")
    .first()
)

best_cloud = best_scene.get(
    "CLOUDY_PIXEL_PERCENTAGE"
).getInfo()

best_date = ee.Date(
    best_scene.get("system:time_start")
).format("YYYY-MM-dd").getInfo()

print("วันที่:", best_date)
print("Cloud percentage:", round(best_cloud, 2))

วันที่: 2024-12-01
Cloud percentage: 0.0


## Step 18 — แสดงภาพเดียวแบบ True Color

True Color ใช้:

```text
Red   = B4
Green = B3
Blue  = B2
```

Sentinel-2 Surface Reflectance เก็บค่าแบบ scaled integer ดังนั้นในขั้นนี้ใช้ช่วง display ประมาณ 0–3000

In [18]:
rgb_raw_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000,
    "gamma": 1.1,
}

Map = create_course_map()
Map.centerObject(bangkok, 10)

Map.addLayer(
    best_scene,
    rgb_raw_vis,
    "Best single Sentinel-2 scene"
)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677679, 100.62341403243893], controls=(WidgetControl(options=['position', 'transparent…

### Observe the Map

สังเกต:
- มี cloud หรือ haze หลงเหลือหรือไม่?
- scene ครอบคลุมกรุงเทพฯ ทั้งหมดหรือไม่?
- พื้นที่น้ำ พื้นที่ก่อสร้าง และ vegetation มีลักษณะอย่างไร?

นี่คือเหตุผลว่าทำไมงาน Remote Sensing มักไม่ใช้ “ภาพเดียว” โดยอัตโนมัติ

## Step 19 — False Color

False Color ชุดง่ายที่ใช้บ่อยกับ vegetation:

```text
Red display   = B8 (NIR)
Green display = B4 (Red)
Blue display  = B3 (Green)
```

vegetation จะเด่นขึ้นกว่าการดู True Color

In [19]:
false_raw_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0,
    "max": 3500,
    "gamma": 1.1,
}

Map = create_course_map()
Map.centerObject(bangkok, 10)

Map.addLayer(
    best_scene,
    rgb_raw_vis,
    "True Color",
    True
)

Map.addLayer(
    best_scene,
    false_raw_vis,
    "False Color",
    False
)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677652, 100.62341403243896], controls=(WidgetControl(options=['position', 'transparent…

# Part E — Cloud Mask และ Composite

## Step 20 — Cloud Mask แบบ Guided Code

การกรอง `CLOUDY_PIXEL_PERCENTAGE` เป็นการคัด scene แต่ใน scene ที่ผ่านเกณฑ์ยังอาจมี cloud บางส่วน

เราจึงใช้ `SCL` (Scene Classification Layer) เพื่อ mask class ที่เกี่ยวข้องกับ:

- cloud shadow
- cloud probability
- cirrus
- snow/ice

> ใน Notebook นี้ให้ใช้ function สำเร็จรูปก่อน ไม่ต้องจำ logic ภายใน

In [20]:
def mask_s2_sr(image):
    scl = image.select("SCL")

    clear = (
        scl.neq(3)      # Cloud shadow
        .And(scl.neq(8))   # Cloud medium probability
        .And(scl.neq(9))   # Cloud high probability
        .And(scl.neq(10))  # Thin cirrus
        .And(scl.neq(11))  # Snow / ice
    )

    # แปลง spectral bands จาก scaled integer เป็น surface reflectance 0-1
    spectral = image.select("B.*").multiply(0.0001)

    return (
        spectral
        .updateMask(clear)
        .copyProperties(image, ["system:time_start"])
    )

## Step 21 — Apply cloud mask กับ ImageCollection

`.map(function)` หมายถึงใช้ function เดียวกันกับทุก image ใน collection

In [21]:
s2_bkk_clean = s2_bkk_cloud.map(
    mask_s2_sr
)

print("จำนวนภาพยังเท่าเดิม:",
      s2_bkk_clean.size().getInfo())

จำนวนภาพยังเท่าเดิม: 34


## Step 22 — สร้าง Median Composite

`median()` รวมข้อมูลหลายเวลาโดยหาค่ามัธยฐานในแต่ละ pixel

ข้อดี:
- ลดอิทธิพลของค่าผิดปกติ
- ลดผลของ cloud/haze บางส่วนเมื่อใช้ร่วมกับ cloud mask

> **ข้อควรระวัง:** median composite ไม่ได้หมายความว่า cloud-free 100% โดยอัตโนมัติ

In [22]:
s2_median = s2_bkk_clean.median()

print(s2_median)

ee.Image({
  "functionInvocationValue": {
    "functionName": "reduce.median",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.map",
          "arguments": {
            "baseAlgorithm": {
              "functionDefinitionValue": {
                "argumentNames": [
                  "_MAPPING_VAR_0_0"
                ],
                "body": {
                  "functionInvocationValue": {
                    "functionName": "Image.copyProperties",
                    "arguments": {
                      "destination": {
                        "functionInvocationValue": {
                          "functionName": "Image.updateMask",
                          "arguments": {
                            "image": {
                              "functionInvocationValue": {
                                "functionName": "Image.multiply",
                                "arguments": {
                                  "

## Step 23 — Clip ตามขอบเขตกรุงเทพมหานคร

`clip()` ตัด raster ให้เหลือเฉพาะ ROI

แนวคิดนี้เหมือนการ Clip Raster ด้วย polygon ใน Desktop GIS

In [23]:
s2_bangkok = s2_median.clip(
    bangkok_roi
)

print(s2_bangkok)

ee.Image({
  "functionInvocationValue": {
    "functionName": "Image.clip",
    "arguments": {
      "geometry": {
        "functionInvocationValue": {
          "functionName": "Collection.geometry",
          "arguments": {
            "collection": {
              "functionInvocationValue": {
                "functionName": "Collection.filter",
                "arguments": {
                  "collection": {
                    "functionInvocationValue": {
                      "functionName": "Collection.filter",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.loadTable",
                            "arguments": {
                              "tableId": {
                                "constantValue": "FAO/GAUL/2025/level1"
                              }
                            }
                          }
                        },
  

## Step 24 — แสดง Composite แบบ True Color และ False Color

หลัง scale reflectance แล้ว ใช้ช่วงแสดงผลประมาณ 0.02–0.30

In [24]:
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.02,
    "max": 0.30,
    "gamma": 1.1,
}

false_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0.02,
    "max": 0.35,
    "gamma": 1.1,
}

Map = create_course_map()
Map.centerObject(bangkok, 10)

Map.addLayer(
    s2_bangkok,
    rgb_vis,
    "Sentinel-2 True Color",
    True
)

Map.addLayer(
    s2_bangkok,
    false_vis,
    "Sentinel-2 False Color",
    False
)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677652, 100.62341403243896], controls=(WidgetControl(options=['position', 'transparent…

### Checkpoint — อ่านภาพ

1. แม่น้ำเจ้าพระยาปรากฏอย่างไรใน True Color?
2. vegetation เด่นขึ้นอย่างไรใน False Color?
3. พื้นที่ built-up หนาแน่นมี pattern อย่างไร?
4. การ clip ช่วยให้การอ่านพื้นที่ศึกษาชัดเจนขึ้นอย่างไร?

# Part F — ดัชนีอย่างง่าย: NDVI

## Step 25 — แนวคิด NDVI

NDVI ใช้ความแตกต่างระหว่าง Near Infrared (NIR) และ Red:

$$
NDVI = \frac{NIR - Red}{NIR + Red}
$$

สำหรับ Sentinel-2:

```text
NIR = B8
Red = B4
```

โดยทั่วไป vegetation ที่เขียวและหนาแน่นมักมี NDVI เป็นบวกค่อนข้างสูง

> ไม่ควรใช้ threshold เดียวแบบตายตัวเพื่อจำแนก vegetation ทุกพื้นที่และทุกฤดูกาล

In [25]:
ndvi = (
    s2_bangkok
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

print(ndvi)

ee.Image({
  "functionInvocationValue": {
    "functionName": "Image.rename",
    "arguments": {
      "input": {
        "functionInvocationValue": {
          "functionName": "Image.normalizedDifference",
          "arguments": {
            "bandNames": {
              "constantValue": [
                "B8",
                "B4"
              ]
            },
            "input": {
              "functionInvocationValue": {
                "functionName": "Image.clip",
                "arguments": {
                  "geometry": {
                    "functionInvocationValue": {
                      "functionName": "Collection.geometry",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "Collection.filter",
                            "arguments": {
                              "collection": {
                                "functionInvocationValue": {
  

## Step 26 — แสดง NDVI บนแผนที่

ใช้สีแบบ sequential จากค่าต่ำไปสูง

In [26]:
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#d73027",
        "#fee08b",
        "#ffffbf",
        "#d9ef8b",
        "#1a9850"
    ],
}

Map = create_course_map()
Map.centerObject(bangkok, 10)

Map.addLayer(
    ndvi,
    ndvi_vis,
    "Bangkok NDVI"
)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map.add_colorbar(
    ndvi_vis,
    label="NDVI",
    layer_name="Bangkok NDVI"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677679, 100.62341403243893], controls=(WidgetControl(options=['position', 'transparent…

### Exercise 3 — อ่าน NDVI จากแผนที่

เลือกดูด้วยสายตา:

- พื้นที่ NDVI สูง
- พื้นที่ NDVI ต่ำ
- แม่น้ำ/แหล่งน้ำ

**คำถาม:** pattern ของ NDVI สอดคล้องกับ basemap satellite imagery หรือไม่?

# Part G — อ่านค่าจากตำแหน่งบนแผนที่

## Step 27 — Query ค่า NDVI ณ 1 ตำแหน่ง

ตัวอย่างใช้จุดบริเวณสวนลุมพินี

หลักการ:

```text
Point
 ↓
NDVI raster
 ↓
Pixel / local value
```

`reduceRegion()` เป็นคำสั่งสำคัญสำหรับสรุป raster ภายใน geometry

In [27]:
lumphini = ee.Geometry.Point(
    [100.5417, 13.7314]
)

lumphini_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=lumphini,
    scale=10
).get("NDVI")

print(
    "NDVI near Lumphini Park:",
    round(lumphini_ndvi.getInfo(), 3)
)

NDVI near Lumphini Park: 0.759


## Step 28 — แสดงจุดตัวอย่างบน NDVI map

In [28]:
Map = create_course_map()
Map.centerObject(bangkok, 11)

Map.addLayer(
    ndvi,
    ndvi_vis,
    "Bangkok NDVI"
)

Map.addLayer(
    lumphini,
    {"color": "cyan"},
    "Lumphini sample point"
)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677679, 100.62341403243893], controls=(WidgetControl(options=['position', 'transparent…

## Step 29 — เปรียบเทียบหลายตำแหน่ง

เราจะเปรียบเทียบตัวอย่าง 3 ลักษณะพื้นที่:

1. Park
2. Dense urban
3. Peri-urban

จุดเหล่านี้ใช้เพื่อการสาธิตแนวคิด ไม่ได้เป็น sampling design สำหรับงานวิจัย

In [29]:
sample_locations = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Point([100.5417, 13.7314]),
        {"site": "Park"}
    ),
    ee.Feature(
        ee.Geometry.Point([100.5650, 13.7480]),
        {"site": "Dense urban"}
    ),
    ee.Feature(
        ee.Geometry.Point([100.8500, 13.7500]),
        {"site": "Peri-urban"}
    ),
])

site_samples = ndvi.sampleRegions(
    collection=sample_locations,
    properties=["site"],
    scale=10,
    geometries=True
)

site_info = site_samples.getInfo()["features"]

site_rows = [
    {
        "site": f["properties"]["site"],
        "NDVI": f["properties"]["NDVI"]
    }
    for f in site_info
]

pd.DataFrame(site_rows)

,site,NDVI
0,Park,0.758845
1,Dense urban,0.124000
2,Peri-urban,0.453190


### Checkpoint

- จุดทั้ง 3 มี NDVI เท่ากันหรือไม่?
- ถ้าไม่เท่ากัน เหตุใด?
- ค่า NDVI อย่างเดียวเพียงพอที่จะสรุป land use หรือไม่?

# Part H — Random Sampling

## Step 30 — สุ่ม 20 จุดภายในกรุงเทพมหานคร

Random points ช่วยให้เข้าใจแนวคิด sampling จากพื้นที่

ใน Notebook นี้ใช้เพียง 20 จุดเพื่อให้เห็นผลชัดและประมวลผลเร็ว

In [30]:
random_points = ee.FeatureCollection.randomPoints(
    bangkok_roi,
    20,
    42
)

print("จำนวน random points:",
      random_points.size().getInfo())

จำนวน random points: 20


## Step 31 — แสดง Random Points บน NDVI map

In [31]:
random_style = random_points.style(
    color="00FFFF",
    pointSize=5
)

Map = create_course_map()
Map.centerObject(bangkok, 10)

Map.addLayer(
    ndvi,
    ndvi_vis,
    "Bangkok NDVI"
)

Map.addLayer(
    random_style,
    {},
    "20 random points"
)

Map.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map.add_layer_control()
Map

Map(center=[13.771850945677652, 100.62341403243896], controls=(WidgetControl(options=['position', 'transparent…

## Step 32 — ดึงค่า NDVI ที่ Random Points

เรากำลังเปลี่ยน:

```text
Raster
  ↓
Sample points
  ↓
Table
```

นี่คือสะพานสำคัญจาก GIS/Remote Sensing ไปสู่ Data Analysis

In [32]:
random_samples = ndvi.sampleRegions(
    collection=random_points,
    scale=10,
    geometries=False
)

sample_info = random_samples.getInfo()["features"]

sample_rows = [
    {
        "point_id": i + 1,
        "NDVI": f["properties"]["NDVI"]
    }
    for i, f in enumerate(sample_info)
]

sample_df = pd.DataFrame(sample_rows)

sample_df.head(10)

,point_id,NDVI
0,1,0.528790
1,2,0.460238
2,3,-0.560589
3,4,0.118105
4,5,0.119938
5,6,0.812386
6,7,0.426303
7,8,0.679322
8,9,0.697853
9,10,0.179429


## Step 33 — ดูสถิติง่าย ๆ ของ Sample

ใน Notebook แรกเรายังไม่สร้าง model เพียงดูว่า sample มีความแปรผันเพียงใด

In [33]:
sample_df["NDVI"].describe()

,NDVI
count,20.000000
mean,0.326813
std,0.298253
min,-0.560589
25%,0.183187
50%,0.276691
75%,0.537551
max,0.812386


### Exercise 4 — เปลี่ยนจำนวนจุด

ลองเปลี่ยนจาก:

```python
20
```

เป็น:

```python
50
```

แล้วตรวจสอบ:
- จำนวนแถวของตาราง
- ค่า mean NDVI
- ค่า min/max

**คำถาม:** เมื่อจำนวนจุดเพิ่มขึ้น ค่าที่สรุปได้เหมือนเดิมทุกครั้งหรือไม่?

# Part I — สร้าง Final Interactive Map

## Step 34 — รวม Layer ที่สำคัญไว้ในแผนที่เดียว

แผนที่สุดท้ายมี:

- Esri basemaps
- Sentinel-2 True Color
- Sentinel-2 False Color
- NDVI
- Bangkok boundary
- Random sample points

ให้เปิด True Color เป็นค่าเริ่มต้น และเปิด layer อื่นจาก Layer Control เมื่อต้องการเปรียบเทียบ

In [34]:
Map_final = create_course_map()
Map_final.centerObject(bangkok, 10)

Map_final.addLayer(
    s2_bangkok,
    rgb_vis,
    "Sentinel-2 True Color",
    True
)

Map_final.addLayer(
    s2_bangkok,
    false_vis,
    "Sentinel-2 False Color",
    False
)

Map_final.addLayer(
    ndvi,
    ndvi_vis,
    "NDVI",
    False
)

Map_final.addLayer(
    random_style,
    {},
    "Random points",
    False
)

Map_final.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map_final.add_layer_control()
Map_final

Map(center=[13.771850945677652, 100.62341403243896], controls=(WidgetControl(options=['position', 'transparent…

# Part J — Export ข้อมูล

## Step 35 — Export NDVI เป็น GeoTIFF ไปยัง Google Drive

GeoTIFF สามารถนำไปเปิดต่อใน:

- QGIS
- ArcGIS Pro
- Python rasterio
- GIS software อื่น ๆ

Earth Engine Export เป็น **task** จึงไม่ได้ดาวน์โหลดทันทีใน cell เดียว แต่จะสร้างงานส่งออกไปยัง Google Drive

พารามิเตอร์สำคัญ:
- `image` = raster ที่จะส่งออก
- `description` = ชื่องาน
- `folder` = folder ใน Google Drive
- `region` = ขอบเขตพื้นที่
- `scale` = pixel size
- `fileFormat` = GeoTIFF

In [35]:
ndvi_export_task = ee.batch.Export.image.toDrive(
    image=ndvi,
    description="Bangkok_NDVI_Sentinel2",
    folder="GEE_Urban_Studies",
    fileNamePrefix="Bangkok_NDVI_2024_2025",
    region=bangkok_roi,
    scale=10,
    maxPixels=1e10,
    fileFormat="GeoTIFF"
)

ndvi_export_task.start()

print("NDVI export task started.")
print(ndvi_export_task.status())

NDVI export task started.
{'state': 'READY', 'description': 'Bangkok_NDVI_Sentinel2', 'priority': 100, 'creation_timestamp_ms': 1790660735001, 'update_timestamp_ms': 1790660735001, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'FRGXP7EKJF6AWJ4X4O4T4KVL', 'name': 'projects/project-cf559859-d42a-44d4-925/operations/FRGXP7EKJF6AWJ4X4O4T4KVL'}


## Step 36 — Export Random Sample เป็น CSV

นอกจาก raster แล้ว Earth Engine สามารถ export ตารางได้ด้วย

In [36]:
csv_export_task = ee.batch.Export.table.toDrive(
    collection=random_samples,
    description="Bangkok_NDVI_Random_Samples",
    folder="GEE_Urban_Studies",
    fileNamePrefix="Bangkok_NDVI_Random_Samples",
    fileFormat="CSV"
)

csv_export_task.start()

print("CSV export task started.")
print(csv_export_task.status())

CSV export task started.
{'state': 'READY', 'description': 'Bangkok_NDVI_Random_Samples', 'priority': 100, 'creation_timestamp_ms': 1790660740454, 'update_timestamp_ms': 1790660740454, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'CIBIBEVRXSSMJSXEDHVLVQPF', 'name': 'projects/project-cf559859-d42a-44d4-925/operations/CIBIBEVRXSSMJSXEDHVLVQPF'}


## Step 37 — Optional: Export ขอบเขตกรุงเทพฯ เป็น Shapefile

ขั้นนี้ช่วยเชื่อมแนวคิด Earth Engine `FeatureCollection` กลับไปสู่ Desktop GIS

ไฟล์ Shapefile ที่ export จะประกอบด้วยหลายไฟล์ เช่น `.shp`, `.shx`, `.dbf`, `.prj`

In [37]:
shp_export_task = ee.batch.Export.table.toDrive(
    collection=bangkok,
    description="Bangkok_GAUL_Boundary",
    folder="GEE_Urban_Studies",
    fileNamePrefix="Bangkok_GAUL_2025",
    fileFormat="SHP"
)

shp_export_task.start()

print("Shapefile export task started.")
print(shp_export_task.status())

Shapefile export task started.
{'state': 'READY', 'description': 'Bangkok_GAUL_Boundary', 'priority': 100, 'creation_timestamp_ms': 1790660743609, 'update_timestamp_ms': 1790660743609, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'QFSONE4RUTS3MPLIZZ4B5C55', 'name': 'projects/project-cf559859-d42a-44d4-925/operations/QFSONE4RUTS3MPLIZZ4B5C55'}


# Part K — แบบฝึกหัดสรุป

## Mini Exercise — สำรวจกรุงเทพมหานครด้วย Sentinel-2

ให้ทำด้วยตนเองโดยใช้ code ที่เรียนมาแล้ว

### Exercise A — เปลี่ยนช่วงเวลา
เปลี่ยนช่วงเป็น:

```text
2024-03-01 ถึง 2024-05-01
```

แล้วตอบ:
1. มีภาพกี่ scene?
2. จำนวนภาพหลัง cloud filter เปลี่ยนหรือไม่?
3. ภาพ True Color แตกต่างจากช่วงเดิมอย่างไร?

### Exercise B — Cloud threshold
ทดลอง:
- 10%
- 20%
- 40%

บันทึกจำนวนภาพที่เหลือ

### Exercise C — NDVI
เติมคำในช่องว่าง:

```python
ndvi_ex = image.normalizedDifference(
    ["____", "____"]
).rename("NDVI")
```

### Exercise D — Random sampling
เปลี่ยน random points จาก 20 เป็น 30 จุด แล้วสร้างตาราง NDVI

### Exercise E — Export
ส่งออก `ndvi_ex` เป็น GeoTIFF ที่ `scale=10`

# SOLUTIONS — เปิดหลังทำแบบฝึกหัดแล้ว

## Solution A — ช่วงเวลาใหม่และ Cloud Filter

In [38]:
EX_START = "2024-03-01"
EX_END   = "2024-05-01"

exercise_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(bangkok_roi)
    .filterDate(EX_START, EX_END)
)

exercise_cloud20 = exercise_collection.filter(
    ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)
)

print("ก่อน cloud filter:",
      exercise_collection.size().getInfo())

print("หลัง cloud filter < 20%:",
      exercise_cloud20.size().getInfo())

ก่อน cloud filter: 39
หลัง cloud filter < 20%: 18


## Solution B — เปรียบเทียบ Cloud Threshold

In [39]:
for threshold in [10, 20, 40]:
    n = (
        exercise_collection
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                threshold
            )
        )
        .size()
        .getInfo()
    )

    print(
        f"Cloud < {threshold}% : {n} scenes"
    )

Cloud < 10% : 13 scenes
Cloud < 20% : 18 scenes
Cloud < 40% : 22 scenes


## Solution C — NDVI

In [40]:
exercise_clean = exercise_cloud20.map(mask_s2_sr)

exercise_image = (
    exercise_clean
    .median()
    .clip(bangkok_roi)
)

ndvi_ex = (
    exercise_image
    .normalizedDifference(["B8", "B4"])
    .rename("NDVI")
)

Map_ex = create_course_map()
Map_ex.centerObject(bangkok, 10)

Map_ex.addLayer(
    ndvi_ex,
    ndvi_vis,
    "Exercise NDVI"
)

Map_ex.addLayer(
    bangkok_outline,
    {},
    "Bangkok boundary"
)

Map_ex.add_layer_control()
Map_ex

Map(center=[13.771850945677679, 100.62341403243893], controls=(WidgetControl(options=['position', 'transparent…

## Solution D — Random 30 points

In [41]:
random30 = ee.FeatureCollection.randomPoints(
    bangkok_roi,
    30,
    42
)

samples30 = ndvi_ex.sampleRegions(
    collection=random30,
    scale=10,
    geometries=False
)

info30 = samples30.getInfo()["features"]

df30 = pd.DataFrame([
    {
        "point_id": i + 1,
        "NDVI": f["properties"]["NDVI"]
    }
    for i, f in enumerate(info30)
])

print("จำนวน sample:", len(df30))
df30.head()

จำนวน sample: 30


,point_id,NDVI
0,1,0.812784
1,2,0.494023
2,3,-0.401460
3,4,0.063800
4,5,0.353889


## Solution E — Export NDVI Exercise เป็น GeoTIFF

In [42]:
exercise_export = ee.batch.Export.image.toDrive(
    image=ndvi_ex,
    description="Bangkok_NDVI_Exercise",
    folder="GEE_Urban_Studies",
    fileNamePrefix="Bangkok_NDVI_Exercise",
    region=bangkok_roi,
    scale=10,
    maxPixels=1e10,
    fileFormat="GeoTIFF"
)

exercise_export.start()

print(exercise_export.status())

{'state': 'READY', 'description': 'Bangkok_NDVI_Exercise', 'priority': 100, 'creation_timestamp_ms': 1790660770794, 'update_timestamp_ms': 1790660770794, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'D2U66IRCETZA3SOCEYYDOIES', 'name': 'projects/project-cf559859-d42a-44d4-925/operations/D2U66IRCETZA3SOCEYYDOIES'}


# สรุปสิ่งที่เรียนจาก Notebook 01

หลังจบ Notebook นี้ workflow ที่ควรจำไม่ใช่ syntax ทุกบรรทัด แต่คือ:

```text
1. สร้าง Map
2. เลือกพื้นที่ (Geometry / FeatureCollection)
3. เรียก ImageCollection
4. filterBounds()
5. filterDate()
6. กรอง metadata เช่น cloud
7. เลือก Image หรือสร้าง Composite
8. แสดง RGB / False Color
9. clip() ตาม ROI
10. สร้าง NDVI
11. อ่านค่าจาก Point
12. Random sample
13. Raster → Table
14. Export GeoTIFF / CSV / Shapefile
```

### แนวคิดสำคัญที่ควรเข้าใจ
- **Basemap** ไม่ใช่ข้อมูลวิเคราะห์
- **FeatureCollection** คือ vector data บน Earth Engine
- **ImageCollection** คือภาพหลาย scene/หลายเวลา
- `filterBounds()` = เลือกตามพื้นที่
- `filterDate()` = เลือกตามเวลา
- Cloud percentage filter ≠ pixel-level cloud mask
- Median composite ≠ cloud-free image โดยอัตโนมัติ
- NDVI เป็น spectral index ไม่ใช่ land-use class
- Sampling ช่วยเปลี่ยน raster ไปเป็น table
- Export ทำให้ผลจาก GEE ไปใช้ต่อใน GIS/Data Analysis ได้

---

## Dataset หลัก
- FAO GAUL 2025 Level 1
- Sentinel-2 Surface Reflectance Harmonized

## เอกสารประกอบที่ควรอ่านต่อ
- Google Earth Engine Data Catalog
- Earth Engine Python API documentation
- geemap documentation